# Slab Weight Pathway Input Coverage

Analyze ECTP slab coverage for `slab_weight_shear` and `slab_weight_shear_with_elasticity`, then export the paper-ready coverage and attrition figures plus compact tables for the manuscript.


In [1]:
import math
import warnings
warnings.filterwarnings('ignore')

import pandas as pd

from notebook_utils import create_ectp_slabs, hess_rcparams, load_pits
from paper_figure_utils import (
    build_slab_weight_coverage_comparison_figure,
    build_slab_weight_shear_with_elasticity_attrition_figure,
    format_method_path,
    notebook_latex,
    prepare_slab_weight_shear_table,
    prepare_slab_weight_shear_with_elasticity_table,
    save_paper_figure,
)
from snowpyt_mechparams.pathway import find_parameterizations
from snowpyt_mechparams.execution import ExecutionEngine
from snowpyt_mechparams.graph import default_graph

try:
    from tqdm import tqdm
except ImportError:
    def tqdm(items, **_kwargs):
        return items

hess_rcparams()


## Load Data and Enumerate Paths

`slab_weight_shear` coverage is evaluated from density, layer thickness, and slope angle. `slab_weight_shear_with_elasticity` coverage adds elastic modulus and Poisson's ratio availability on every slab layer.


In [2]:
def count_ok(traces, param: str, n_layers: int) -> bool:
    return sum(
        1
        for trace in traces
        if trace.parameter == param
        and trace.layer_index is not None
        and trace.success
        and trace.output is not None
    ) == n_layers


def has_finite_value(value) -> bool:
    if value is None:
        return False
    value = getattr(value, 'n', value)
    try:
        return math.isfinite(float(value))
    except (TypeError, ValueError):
        return False


pits = load_pits()
ectp_slabs = create_ectp_slabs(pits)
total_slabs = len(ectp_slabs)

engine = ExecutionEngine()
shear_paths = find_parameterizations(default_graph, default_graph.get_node('slab_weight_shear'))
elasticity_paths = find_parameterizations(default_graph, default_graph.get_node('slab_weight_shear_with_elasticity'))

print(f'Loaded {len(pits):,} pits and {total_slabs:,} ECTP slabs')
print(f'slab_weight_shear pathways: {len(shear_paths)}')
print(f'slab_weight_shear_with_elasticity pathways: {len(elasticity_paths)}')


Loaded 50,278 pits and 14,776 ECTP slabs
slab_weight_shear pathways: 4
slab_weight_shear_with_elasticity pathways: 32


## Slab Weight_Shear Coverage Summary


In [3]:
shear_records = []

for slab in tqdm(ectp_slabs, desc='slab_weight_shear coverage', unit='slab'):
    results = engine.execute_all(slab, 'slab_weight_shear')
    n_layers = len(slab.layers)
    thickness_ok = all(layer.thickness is not None for layer in slab.layers)
    slope_angle_ok = has_finite_value(slab.angle)

    for pathway in results.pathways.values():
        density_method = pathway.methods_used.get('density', 'data_flow')
        ok_density = count_ok(pathway.computation_trace, 'density', n_layers)
        shear_records.append(
            {
                'slab_id': slab.slab_id,
                'density_method': density_method,
                'slab_density_ok': ok_density,
                'thickness_ok': thickness_ok,
                'slope_angle_ok': slope_angle_ok,
                'all_inputs_ok': ok_density and thickness_ok and slope_angle_ok,
            }
        )

shear_df = pd.DataFrame(shear_records)
shear_cov = (
    shear_df.groupby('density_method')
    .agg(
        n_density_ok=('slab_density_ok', 'sum'),
        n_thickness_ok=('thickness_ok', 'sum'),
        n_slope_angle_ok=('slope_angle_ok', 'sum'),
        n_all_inputs=('all_inputs_ok', 'sum'),
    )
    .reset_index()
    .sort_values('n_all_inputs', ascending=False)
)
shear_table = prepare_slab_weight_shear_table(shear_cov, total_slabs)
shear_table


slab_weight_shear coverage:   0%|          | 0/14776 [00:00<?, ?slab/s]

slab_weight_shear coverage:   3%|▎         | 422/14776 [00:00<00:03, 4212.87slab/s]

slab_weight_shear coverage:   6%|▌         | 844/14776 [00:00<00:03, 4209.96slab/s]

slab_weight_shear coverage:   9%|▊         | 1286/14776 [00:00<00:03, 4302.20slab/s]

slab_weight_shear coverage:  12%|█▏        | 1767/14776 [00:00<00:02, 4499.57slab/s]

...

slab_weight_shear coverage:  93%|█████████▎| 13705/14776 [00:03<00:00, 4528.64slab/s]

slab_weight_shear coverage:  96%|█████████▌| 14168/14776 [00:03<00:00, 4556.13slab/s]

slab_weight_shear coverage:  99%|█████████▉| 14624/14776 [00:03<00:00, 4482.66slab/s]

slab_weight_shear coverage: 100%|██████████| 14776/14776 [00:03<00:00, 4324.69slab/s]

,Density method,Successful slabs,Coverage (%)
2,Kim and Jamieson Table 2,"5,470",37.0
1,Geldsetzer and Jamieson (2000),"4,187",28.3
3,Kim and Jamieson Table 5,697,4.7
0,Direct matched density,109,0.7


## Slab Weight_Shear With Elasticity Coverage, Attrition, and Figure Export


In [4]:
elasticity_records = []

for slab in tqdm(ectp_slabs, desc='slab_weight_shear_with_elasticity coverage', unit='slab'):
    results = engine.execute_all(slab, 'slab_weight_shear_with_elasticity')
    n_layers = len(slab.layers)
    thickness_ok = all(layer.thickness is not None for layer in slab.layers)
    slope_angle_ok = has_finite_value(slab.angle)

    for pathway in results.pathways.values():
        methods = pathway.methods_used
        density_method = methods.get('density', 'data_flow')
        emod_method = methods.get('elastic_modulus', '?')
        nu_method = methods.get('poissons_ratio', '?')

        ok_density = count_ok(pathway.computation_trace, 'density', n_layers)
        ok_emod = count_ok(pathway.computation_trace, 'elastic_modulus', n_layers)
        ok_nu = count_ok(pathway.computation_trace, 'poissons_ratio', n_layers)

        elasticity_records.append(
            {
                'slab_id': slab.slab_id,
                'density_method': density_method,
                'emod_method': emod_method,
                'nu_method': nu_method,
                'ok_density': ok_density,
                'ok_thickness': thickness_ok,
                'ok_slope_angle': slope_angle_ok,
                'ok_emod': ok_emod,
                'ok_nu': ok_nu,
                'all_inputs_ok': ok_density and thickness_ok and slope_angle_ok and ok_emod and ok_nu,
            }
        )

elasticity_df = pd.DataFrame(elasticity_records)
elasticity_cov = (
    elasticity_df.groupby(['density_method', 'emod_method', 'nu_method'])
    .agg(
        n_density_ok=('ok_density', 'sum'),
        n_thickness_ok=('ok_thickness', 'sum'),
        n_slope_angle_ok=('ok_slope_angle', 'sum'),
        n_emod_ok=('ok_emod', 'sum'),
        n_nu_ok=('ok_nu', 'sum'),
        n_all_inputs=('all_inputs_ok', 'sum'),
    )
    .reset_index()
    .sort_values('n_all_inputs', ascending=False)
)

best_density = 'kim_jamieson_table2'
best_emod = 'wautier'
best_nu = 'kochle'
best_subset = elasticity_df[
    (elasticity_df['density_method'] == best_density)
    & (elasticity_df['emod_method'] == best_emod)
    & (elasticity_df['nu_method'] == best_nu)
].copy()

ok_shear_inputs = (
    best_subset['ok_density']
    & best_subset['ok_thickness']
    & best_subset['ok_slope_angle']
)
attrition_steps = [
    ('', total_slabs),
    (r'$\rho$ valid', int(best_subset['ok_density'].sum())),
    (r'$\rho + h_i + \psi$ valid', int(ok_shear_inputs.sum())),
    (r'$\rho + h_i + \psi + E$ valid', int((ok_shear_inputs & best_subset['ok_emod']).sum())),
    (r'$\rho + h_i + \psi + E + \nu$ valid', int(best_subset['all_inputs_ok'].sum())),
]
attrition_methods = [
    '',
    r'$\rho$ method: Kim / Jamieson T2',
    r'$h_i$, $\psi$ valid',
    r'$E$ method: Wautier',
    r'$\nu$ method: Kochle',
]

coverage_paths = save_paper_figure(
    build_slab_weight_coverage_comparison_figure(shear_cov, elasticity_cov, total_slabs, top_n=10),
    'slab_weight_coverage_comparison',
    close=True,
)
attrition_paths = save_paper_figure(
    build_slab_weight_shear_with_elasticity_attrition_figure(
        attrition_steps,
        total_slabs,
        format_method_path(best_density, best_emod, best_nu, short=True),
        method_steps=attrition_methods,
    ),
    'slab_weight_shear_with_elasticity_attrition',
    close=True,
)

elasticity_table = prepare_slab_weight_shear_with_elasticity_table(elasticity_cov, total_slabs, top_n=8)
print('Coverage figure exports:', coverage_paths)
print('Attrition figure exports:', attrition_paths)
elasticity_table


slab_weight_shear_with_elasticity coverage:   0%|          | 0/14776 [00:00<?, ?slab/s]

slab_weight_shear_with_elasticity coverage:   0%|          | 38/14776 [00:00<00:38, 377.92slab/s]

slab_weight_shear_with_elasticity coverage:   1%|          | 76/14776 [00:00<01:50, 132.47slab/s]

slab_weight_shear_with_elasticity coverage:   1%|          | 110/14776 [00:00<01:21, 179.12slab/s]

slab_weight_shear_with_elasticity coverage:   1%|          | 148/14776 [00:00<01:04, 227.56slab/s]

...

'created' timestamp seems very low; regarding as unix timestamp


1 extra bytes in post.stringData array


'created' timestamp seems very low; regarding as unix timestamp


Coverage figure exports: {'repo_pdf': PosixPath('/Users/marykate/Desktop/Snow/SnowPyt-MechParams/paper/figures/slab_weight_coverage_comparison.pdf'), 'repo_png': PosixPath('/Users/marykate/Desktop/Snow/SnowPyt-MechParams/paper/figures/slab_weight_coverage_comparison.png'), 'external_pdf': PosixPath('/Users/marykate/Desktop/Snow/mech_params_paper/figures/slab_weight_coverage_comparison.pdf'), 'external_png': PosixPath('/Users/marykate/Desktop/Snow/mech_params_paper/figures/slab_weight_coverage_comparison.png'), 'pdf': PosixPath('/Users/marykate/Desktop/Snow/SnowPyt-MechParams/paper/figures/slab_weight_coverage_comparison.pdf'), 'png': PosixPath('/Users/marykate/Desktop/Snow/SnowPyt-MechParams/paper/figures/slab_weight_coverage_comparison.png'), 'external_dir': PosixPath('/Users/marykate/Desktop/Snow/mech_params_paper/figures')}
Attrition figure exports: {'repo_pdf': PosixPath('/Users/marykate/Desktop/Snow/SnowPyt-MechParams/paper/figures/slab_weight_shear_with_elasticity_attrition.pdf')

,Density method,E method,nu method,Successful slabs,Coverage (%)
22,Kim and Jamieson Table 2,Wautier et al. (2015),Kochle and Schneebeli (2014),687,4.6
20,Kim and Jamieson Table 2,Schottner et al. (2026),Kochle and Schneebeli (2014),687,4.6
12,Geldsetzer and Jamieson (2000),Schottner et al. (2026),Kochle and Schneebeli (2014),684,4.6
14,Geldsetzer and Jamieson (2000),Wautier et al. (2015),Kochle and Schneebeli (2014),684,4.6
10,Geldsetzer and Jamieson (2000),Kochle and Schneebeli (2014),Kochle and Schneebeli (2014),677,4.6
18,Kim and Jamieson Table 2,Kochle and Schneebeli (2014),Kochle and Schneebeli (2014),581,3.9
16,Kim and Jamieson Table 2,Bergfeld et al. (2023),Kochle and Schneebeli (2014),465,3.1
8,Geldsetzer and Jamieson (2000),Bergfeld et al. (2023),Kochle and Schneebeli (2014),443,3.0


## Copy-Ready LaTeX Tables


In [5]:
print('slab_weight_shear table:')
print(notebook_latex(shear_table))
print()
print('slab_weight_shear_with_elasticity table:')
print(notebook_latex(elasticity_table))


slab_weight_shear table:
\begin{tabular}{lll}
\toprule
Density method & Successful slabs & Coverage (%) \\
\midrule
Kim and Jamieson Table 2 & 5,470 & 37.0 \\
Geldsetzer and Jamieson (2000) & 4,187 & 28.3 \\
Kim and Jamieson Table 5 & 697 & 4.7 \\
Direct matched density & 109 & 0.7 \\
\bottomrule
\end{tabular}


slab_weight_shear_with_elasticity table:
\begin{tabular}{lllll}
\toprule
Density method & E method & nu method & Successful slabs & Coverage (%) \\
\midrule
Kim and Jamieson Table 2 & Wautier et al. (2015) & Kochle and Schneebeli (2014) & 687 & 4.6 \\
Kim and Jamieson Table 2 & Schottner et al. (2026) & Kochle and Schneebeli (2014) & 687 & 4.6 \\
Geldsetzer and Jamieson (2000) & Schottner et al. (2026) & Kochle and Schneebeli (2014) & 684 & 4.6 \\
Geldsetzer and Jamieson (2000) & Wautier et al. (2015) & Kochle and Schneebeli (2014) & 684 & 4.6 \\
Geldsetzer and Jamieson (2000) & Kochle and Schneebeli (2014) & Kochle and Schneebeli (2014) & 677 & 4.6 \\
Kim and Jamieson Table 2 